# Reductions, Atomics and Shared Memory

We start, as usual, by loading our ice magic.

In [ ]:
%load_ext ice.magic

## Reductions

The last computational pattern this course discussed is that of a **reduction**.
In reductions, each CUDA thread produces one value and all separate contributions need to be consolidated into one (or more) values.
Prominent examples in practice include
* computing the length of a vector,
* computing the dot product of two vectors,
* finding the maximum or minimum value in an array, or
* constructing a histogram.

## A Naive Implementation

The below code implements a naive **sum reduction**, i.e. all thread contributions are summed up into a global accumulator.
To make the example as simple as possible, each thread adds `1` for each 'work element' it is responsible for.

In [ ]:
%%cuda -c code/reduction/naive.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < numElements; i += stride)
        acc[0] += 1;
           👆    👆
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

## Atomic Operations

<img align="right" src="img/reduction-race-cond.png" alt="Reduction Race Condition" width="640px"  style="background-color:white"/>

Running the code above reveals a fundamental flaw with its naive implementation: it is not **thread-safe**.
Since multiple threads are updating the same data element potentially concurrently, a **race condition** occurs.

<img align="right" src="img/reduction-atomic.png" alt="Reduction Atomic Add" width="640px"  style="background-color:white"/>

There are different ways of fixing this issue.
The most straight-forward one is using **atomic operations** that guarantee that each memory location's update is carried out by only a single thread at each given time.

```c++
T atomicAdd(T* address, T value);
```
* Atomically adds a given `value` to the data at a given `address`.
* Returns the old content of `address` (before updating).
* Supports different data types `T`.
* [Documentation](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/cpp-language-extensions.html#atomicadd)

Note that variants for other operation types exist, e.g. `atomicInc`, `atomicMin`, `atomicExch`, ...

Also note that the functions discussed so far are **legacy atomic functions**.
In C++, additional **synchronization primitives** are available as part of `libcu++` ([Documentation](https://nvidia.github.io/cccl/unstable/libcudacxx/extended_api/synchronization_primitives.html))

## Exercise: Add Atomic Operations

The below code is a copy of the naive implementation discussed previously.

* Make the kernel thread-safe by using atomics.
* Verify that the result of the operations is now correct.
* Compare the performance of this version with the naive (and defective) implementation.

In [ ]:
%%cuda -c code/reduction/ex-atomic.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < numElements; i += stride)
        acc[0] += 1;
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

### Possible Solution

In [ ]:
%%cuda -c code/reduction/sol-atomic.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < numElements; i += stride)
        //# acc[0] += 1;
        atomicAdd(&acc[0], 1);
              👆       👆      👆
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

The performance drops from about 250 GFLOP/s to about 25 GFLOP/s (a 10x slowdown) when using atomic operations, but now the result is correct.

## Hierarchical Reductions - Thread Level

As we've seen in the last exercise, our reduction is now correct, but its performance is quite low (10x slower than the *incorrect* baseline).

The main result lies in a high degree of **atomic congestion**.
This effect usually occurs when many threads update the **same memory location**.

To remedy this issue, it is paramount to reduce the number of (parallel) atomic operations into the global accumulator.
There are many different options to do this - duplicate the accumulator, multi-step reductions, and local accumulations.
We start with the latter by introducing a thread-local accumulators.

In combination with the employed grid-stride loop, the number of atomics is reduced from the number of 'work elements' (4 194 304) to the number of threads (688 128).

In [ ]:
%%cuda -c code/reduction/thread-accumulator.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    int thread_acc = 0;

    for (int i = start; i < numElements; i += stride)
        thread_acc += 1;

    atomicAdd(&acc[0], thread_acc);
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

Running the improved version shows a significant performance increase (from 25 GFLOP/s to about 200 GFLOP/s) - a 8x speedup compared to the atomic version and only 1.25x slower than the incorrect baseline.

## Hierarchical Reductions - Block Level

Reducing the number of atomic updates to the total number of threads worked well.
This motivates implementing another level of **hierarchical reductions** with the aim of having only a **single atomic update per block**.
Our approach is the following:
1. Each thread adds into its thread-local accumulator (as before).
2. All threads of one block synchronize their partial results into a block-local result.
3. The first thread of each block adds the block-local result to the global accumulator.

Steps one and three are straight-forward with the knowledge gained in this course so far.
For step two, different approaches are possible.
* Allocating a new device array with as many elements as there are blocks.
  Each element represents one block-local accumulator.
* Allocating **shared memory** to reduce the results within one block.

Since the former approach introduces unnecessary strain on the device memory (actually the L2 cache), we will investigate the latter one.

We start by looking into what shared memory is and how it can be used.

Each SM has a small amount of exclusive memory which is usually used for caching data (L1 cache).
The L1 cache can not be programmed explicitly.
Since threads need to store temporary data in many application scenarios, manual allocations into the same physical memory are supported.
These allocations
* are shared between all threads of a block,
* must have a compile-time constant size, and
* support atomic operations.

Note that in practice, the available memory is split between L1 cache and shared memory.

To allocate shared memory, the `__shared__` keyword is used.

```c++
//# allocate 16 int (block size) ints per block in shared memory
__shared__ int block_mem[16];

//# initialize the shared memory with the first 16 threads
if (threadIdx.x < 16)
    block_mem[threadIdx.x] = thread_acc;

//# something is missing here

//# the first thread is responsible for reducing the shared memory values
if (0 == threadIdx.x) {
    int block_acc = 0.f;
    for (int i = 0; i < 16; ++i)
        block_acc += block_mem[i];
    
    cuda::atomicAdd(&acc[0], block_acc);
}
```

The code above has another race condition - thread 0 might read the values from `block_mem` before they are written *by another thread*.
Atomic operations alone won't solve this issue this time.
Fortunately, we can make use of additional **synchronization primitives**.

```c++
__syncthreads();
```
* Implements a barrier-type operation which synchronizes all threads **within the current block**.

The below code shows an updated example using shared memory to make the thread-local partial results available to other threads within the same block.

In [ ]:
%%cuda -c code/reduction/shared-memory.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    int thread_acc = 0;

    for (int i = start; i < numElements; i += stride)
        thread_acc += 1;

    //# allocate 256 (block size) int per block in shared memory
    __shared__ int block_mem[256];

    //# store the thread-local contributions
    block_mem[threadIdx.x] = thread_acc;

    //# synchronize threads in the block
    __syncthreads();

    //# the first thread is responsible for reducing the shared memory values
    if (0 == threadIdx.x) {
        int block_acc = 0;
        for (int i = 0; i < 256; ++i)
            block_acc += block_mem[i];
        
        atomicAdd(&acc[0], block_acc);
    }
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

This updated version of the code reaches around 320 GFLOP/s - 1.6x faster than the version without shared memory.
It is now also faster than the incorrect baseline (250 GFLOP/s).

The main reason for the improvement is fewer update operations into the global accumulator, and thereby the global memory.
While the reduction of traffic to global memory scales with the block size (256 in our case), the performance improvement is not as pronounced.

## Exercise: Use Shared Memory Atomics

The above code version does a great job at decreasing the atomic congestion and thereby boosting performance.
It still allocates more shared memory than strictly necessary and serializes part of the reductions within one block.
Finish the code below and address these points.

* Allocate only a **single element** per block in shared memory.
* Initialize this block accumulator with 0 using only one thread.
* Let each thread add its contribution to the block accumulator.
  Make sure that this operation is thread-safe.
* Use one thread to update the **global accumulator**.

After finishing your implementation, and verifying that it still produces correct results, compare the performance with the previous version.

In [ ]:
%%cuda -c code/reduction/ex-shared-atomics.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    int thread_acc = 0;

    for (int i = start; i < numElements; i += stride)
        thread_acc += 1;

    //# TODO: allocate a single int per block in shared memory

    //# TODO: initialize block_acc with zero using a single thread

    //# TODO: synchronize threads in the block to wait for the initialization

    //# TODO: add thread_acc into block_acc in a thread-safe way

    //# TODO: synchronize threads in the block

    //# TODO: add block_acc into global acc in a thread-safe way using a single thread
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

### Possible Solution

In [ ]:
%%cuda -c code/reduction/sol-shared-atomics.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    int thread_acc = 0;

    for (int i = start; i < numElements; i += stride)
        thread_acc += 1;

    //# allocate shared memory
    __shared__ int block_acc[1];

    //# initialize shared memory
    if (0 == threadIdx.x)
        block_acc[0] = 0;

    //# synchronize threads in the block
    __syncthreads();

    //# accumulate to shared memory
    atomicAdd(&block_acc[0], thread_acc);

    //# synchronize threads in the block
    __syncthreads();

    //# accumulate block result to global memory
    if (0 == threadIdx.x)
        atomicAdd(&acc[0], block_acc[0]);
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

Improving the implementation to have less serialization **within one thread** leads to another small performance increase (from 320 GFLOP/s to about 360 GFLOP/s).

## Hierarchical Reductions - CUB

At this point we could further optimize our kernel implementation through

* using alternative reduction types for select stages (e.g. tree reductions), and
* adding other stages (e.g. warp-local accumulators using warp-level operations such as warp shuffle instructions).
  If you're interested, this, admittedly slightly old, [blog post](https://developer.nvidia.com/blog/faster-parallel-reductions-kepler/) is a great starting point.

Doing so, however, would make our kernel implementation

* more lengthy,
* harder to maintain,
* harder to verify, and
* less portable.

A viable alternative is relying on existing, highly optimized implementations.
For block-level reductions (and many other operations), `cub` (CUDA Unbound) offers ready to use building blocks.

```c++
//# specialize the type with <accumulator data type, number of threads per block>
using BlockReduce = cub::BlockReduce<int, 256>;

//# allocate shared memory for the block reduction
__shared__ typename BlockReduce::TempStorage tempStorage;

//# perform the block reduction with the `Sum` operation
int block_acc = BlockReduce(tempStorage).Sum(thread_acc);
```

Further information is available in the official [documentation](https://nvidia.github.io/cccl/cub/api/classcub_1_1BlockReduce.html), as well as the [source code](https://github.com/NVIDIA/cccl/blob/main/cub/cub/block/block_reduce.cuh).

Caveats:
* All threads must participate in the operation.
* Only thread 0 of the current block receives the result of the reduction.
* As before, the shared memory allocation must be of constant size.

In [ ]:
%%cuda -c code/reduction/cub.cu

#include <cub/cub.cuh>

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    int thread_acc = 0;

    for (int i = start; i < numElements; i += stride)
        thread_acc += 1;

    //# use cub for a block-wide reduction
    using BlockReduce = cub::BlockReduce<int, 256>;
    __shared__ typename BlockReduce::TempStorage tempStorage;
    int block_acc = BlockReduce(tempStorage).Sum(thread_acc);

    //# only the first thread in the current block receives the block-wide result
    //# and adds it to the global accumulator
    if (0 == threadIdx.x) {
        atomicAdd(&acc[0], block_acc);
    }
}

int main(int argc, char *argv[]) {
    int numElements = 4 * 1024 * 1024;

    int acc = 0;

    int *d_acc;
    cudaMalloc(&d_acc, sizeof(int));

    //# warm-up
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    //# reset accumulator
    cudaMemset(d_acc, 0, sizeof(int));

    auto start = std::chrono::steady_clock::now();

    //# run reduction
    reduce<<<84 * 32, 256>>>(d_acc, numElements);

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-9 * numElements / elapsedSeconds.count() << " GFLOP/s\n";

    //# copy data back to host
    cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    cudaFree(d_acc);
}

This final version of the code show comparable performance to the previous one, but is much more concise and potentially easier to maintain.

## Next Step

To continue with the course, head over to the [summary & outlook](./06-summary-outlook.ipynb) notebook.